# 4. Using existing flows: MgO Phonons

In this section, we make use of density functional **perturbation** theory (DFPT) for computing the phonon band structure of magnesium oxide.

Standalone version:

* `make_mgo_phonons.py`.
* `run_mgo_anaddb.py`.
* `plot_mgo_phonons.py`.


In [1]:
from pathlib import Path
import shutil
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from abipy import abilab
import abipy.flowtk as flowtk
abilab.enable_notebook()

%matplotlib inline

import workshop_lib as wlib

## 4.1 Phonons from DFPT

Density-functional perturbation theory (DFPT) gives access to phonons,
Born effective charges and the dielectric tensor without finite
displacements. AbiPy's `PhononFlow` automates the whole workflow: a GS run
that produces the ground-state wavefunctions (`WFK`), followed by one DFPT
task per symmetry-irreducible atomic perturbation and q-point.

`wlib.build_mgo_phonon_flow` builds a simplified, MgO version of this --
coarser `ecut`, k-mesh and q-mesh, for speed. MgO (rocksalt) is strongly
ionic, which makes for a good showcase of the LO-TO splitting below.


In [5]:
wlib.print_source(wlib.build_mgo_phonon_flow)

Notice how we only need to build a ground state input, and the constructor function produces the full workflow for us!

### Build the flow

In [6]:
flow = wlib.build_mgo_phonon_flow(workdir="flow_mgo_phonons")

In [7]:
# Set number of processors
flow = wlib.setup_manager(flow, mpi_procs=8, omp_threads=1)


Here we actually write the files, but first we check that the flow directory doesn't actually exist.
This is not recommended for your script. The example scripts will not overwrite an existing flow directory, because it might contain precious data.
Hence, the user should remove any previous run ('rm -r flow_to_overwrite') before executing the script.


In [8]:
# Remove an existing flow directory from previus run (not recommended)
if Path(flow.workdir).exists():
    shutil.rmtree(flow.workdir)

In [9]:
# Write the flow to disk
flow.build_and_pickle_dump()

if Path(flow.workdir).exists():
    print(f'Successfully built flow: {flow.workdir}')

Successfully built flow: /Users/antonius/Documents/Workshops/2026-CEMDI/Notebooks/flow_mgo_phonons


### Run the flow

This flow has more tasks than the others (one per symmetry-irreducible
atomic perturbation and q-point), so if you're running the standalone
`make_mgo_phonons.py` from a terminal instead, it's the best candidate for
`nohup python make_mgo_phonons.py > log 2> err &` rather than waiting on
it in a foreground shell. From this notebook, `wlib.shell_command` already
launches it in the background.

In [10]:
wlib.shell_command("abirun.py flow_mgo_phonons scheduler", silent=True)

When you execute this command in the shell, you should see a table that updates every 5s (or whatever time you set in your `~/.abinit/abipy/scheduler.yml` file.)
In this notebook, the scheduler is invoked but the output is suppressed. You can check the status of the workflow with the following command.

In [14]:
# Run this cell several times to see progress
wlib.shell_command('abirun.py flow_mgo_phonons status')

Running on Orpheus -- system Darwin -- Python 3.12.12 -- abirun-0.9.8

Work #0: <Work, node_id=3096, workdir=flow_mgo_phonons/w0>, Finalized=True 
  Finalized works are not shown. Use verbose > 0 to force output.

Work #1: <PhononWork, node_id=3099, workdir=flow_mgo_phonons/w1>, Finalized=True 
  Finalized works are not shown. Use verbose > 0 to force output.

all_ok reached



If you're running the flow in the shell, you will see live updates untill the flow completes.
If you're running it throught this notebook, then you can execute the previous cell several times to see the flow is progressing, until it prints out 'all_ok reached'.

### Examine the result

All the DFPT results (dynamical matrices, Born effective charges,
dielectric tensor) are merged into a single `DDB` file -- the entry point
for essentially all the post-processing below:


In [26]:
ddb = abilab.abiopen("flow_mgo_phonons/w1/outdata/out_DDB")
print(ddb)

================================= File Info =================================
Name: out_DDB
Directory: /Users/antonius/Documents/Workshops/2026-CEMDI/Notebooks/flow_mgo_phonons/w1/outdata
Size: 21.60 kB
Access Time: Tue Jul 14 13:38:55 2026
Modification Time: Tue Jul 14 13:38:53 2026
Change Time: Tue Jul 14 13:38:53 2026

================================= Structure =================================
Full Formula (Mg1 O1)
Reduced Formula: MgO
abc   :   2.965608   2.965608   2.965608
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Mg    0    0    0
  1  O     0.5  0.5  0.5

Abinit Spacegroup: spgid: 0, num_spatial_symmetries: 48, has_timerev: True, symmorphic: False

================================== DDB Info ==================================

Number of q-points in DDB: 3
guessed_ngqpt: [2 2 2] (guess for the q-mesh divisions made by AbiPy)
ecut = 24.000000, ecutsm = 0.000000, nkpt = 36, n

## 4.2 Post-processing with Anaddb

In [3]:
wlib.print_source(wlib.get_anaddb_input)

In [2]:
anaddb_inp = wlib.get_anaddb_input()

In [12]:
# Note that ddb_node is either a file name, a work, or a task producing the DDB.
task = flowtk.AnaddbTask(anaddb_inp, workdir='task_mgo_anaddb', ddb_node="flow_mgo_phonons/w1/outdata/out_DDB")
task = wlib.setup_task_manager(task, mpi_procs=1)

In [13]:
# Remove a previous run, if it exists.
if Path(task.workdir).exists():
    print(f'Removing existing directory: {task.workdir}/')
    task.rmtree()

Removing existing directory: /Users/antonius/Documents/Workshops/2026-CEMDI/Notebooks/task_mgo_anaddb/


In [14]:
# Write files and run the task
task.build()
task.make_links()

In [15]:
# Run it
task.start_and_wait()

0

In [16]:
print(task.check_status())

Completed


In [19]:
# Open phonon band structure
phbst = abilab.abiopen('task_mgo_anaddb/outdata/out_PHBST.nc')

# Open phonon density of state
phdos = abilab.abiopen('task_mgo_anaddb/outdata/out_PHDOS.nc')

In [4]:
phbands = phbst.phbands
#phbands.read_non_anal_from_file('task_mgo_anaddb/outdata/out_anaddb.nc')  # If we include BECS
fig = phbands.plot_with_phdos(phdos, show=True)

NameError: name 'phbst' is not defined

Running anaddb on-the-fly:

`ddb.anaget_phbst_and_phdos_files` calls `anaddb` behind the scenes to
Fourier-interpolate the dynamical matrix onto a dense q-mesh (for the
phonon DOS) and along a high-symmetry q-path (for the phonon band
structure) -- this is exactly what `../Examples/run_mgo_anaddb.py` does,
standalone:


## Challenge

> **Challenge**
> 
> Create a DFPT flow with an extra step running Anaddb to produce the band structure.
> Make sure that the computation of Born effective charges and dielectric tensor are included in the flow (`with_becs=True`)


Continue with [`5-Challenge.ipynb`](5-Challenge.ipynb).